# Лабораторная работа №4: Генерация изображений

[Исходное задание](https://colab.research.google.com/drive/1EE5nJ47nq853xu3AZJQX0JMDOiJbXCai)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

#Задание семинара №4
1. Сгенерировать картинку по вашей теме, используя шаблон промпта для изображений из презентации №2.
2. Есть список из 3-х ключевых слов/направлений. Пользователь пишет запрос на генерацию картинки. Если в запросе есть эти слова, то картинка генерируется, если нет, то об этом сообщается пользователю. Реализовать с помощью 2-х запросов к БЯМ: 1) текстовый запрос на проверку соответствия запроса ключевым словам и 2) генерация картинки при соответствии.
3. Модифицировать решение п.2 таким образом, чтобы выполнялся только один запрос, который вернет либо изображение, либо сообщение, что картинка не соответствует ключевым словам.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. Изображение по шаблону

Шаблон: объект + окружение + действие + стиль + освещение + композиция.

In [ ]:
from lab_runtime import generate_image, request_json
import base64

prompt = "A music producer at a laptop in a recording studio, arranging a song; photorealistic, warm soft lighting, wide composition, no text."
path = generate_image(prompt, "lab04_template.png")
print("Промпт:", prompt)
display(Image(filename=str(path), width=600))


## 2. Проверка трёх ключевых слов и отдельная генерация

In [ ]:
KEYWORDS = ["музыкант", "студия", "ноутбук"]
check_chain = ChatPromptTemplate.from_messages([
    ("system", 'Проверь наличие всех ключевых объектов по смыслу. Верни JSON: {{"match":true/false,"missing":[]}}.'),
    ("human", "Ключевые объекты: {keywords}. Запрос: {request}")
]) | llm | StrOutputParser()

for request in ["Музыкант в студии работает за ноутбуком.", "Кошка гуляет по саду."]:
    checked = json_result(check_chain.invoke({"keywords": KEYWORDS, "request": request}))
    print("Запрос:", request, "Проверка:", checked)
    if checked["match"]:
        generated = generate_image(request + " Фотореалистично, без надписей.", "lab04_keywords.png")
        display(Image(filename=str(generated), width=500))
    else:
        print("Генерация отклонена, отсутствуют:", checked["missing"])


## 3. Один запрос: изображение или отказ

Мультимодальная модель получает условие и выбирает формат результата за один API-вызов.

In [ ]:
import json
single_results = []
for index, request in enumerate(["Музыкант в студии работает за ноутбуком.", "Кошка гуляет по саду."]):
    cache_path = ASSETS / f"lab04_single_{index}.json"
    if cache_path.exists():
        result = json.loads(cache_path.read_text())
    else:
        result = request_json("/chat/completions", {
            "model": "google/gemini-2.5-flash-image",
            "messages": [{"role": "user", "content":
                f"Требуемые объекты: {KEYWORDS}. Запрос: {request}. "
                "Если все объекты есть в запросе, создай картинку. Если хотя бы одного нет, верни только текст отказа, не создавай картинку."}],
            "modalities": ["image", "text"], "max_tokens": 1200
        })
        cache_path.write_text(json.dumps(result, ensure_ascii=False))
    message = result["choices"][0]["message"]
    print("Запрос:", request)
    print("Ответ:", message.get("content"))
    images = message.get("images", [])
    for item in images:
        url = item["image_url"]["url"]
        display(Image(data=base64.b64decode(url.split(",",1)[1]), width=500))
    single_results.append({"request": request, "image_count": len(images), "text": message.get("content")})
assert single_results[0]["image_count"] > 0
assert single_results[1]["image_count"] == 0


## Полученный JSON ответа API

Показываем реальные сохранённые ответы из предыдущего раздела. Новые запросы не отправляются. Только длинные строки Base64 сокращаются для читаемости; полный JSON остаётся в указанном файле.

In [ ]:
import json
from pathlib import Path

# True — напечатать весь JSON, включая несколько мегабайт Base64.
SHOW_FULL_JSON = False

def json_preview(value):
    if isinstance(value, dict):
        return {key: json_preview(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_preview(item) for item in value]
    if isinstance(value, str) and len(value) > 1000:
        return value[:160] + f'... [сокращено для вывода; всего {len(value)} символов]'
    return value

# Находим assets и при самостоятельном запуске этой ячейки.
assets_dir = next(
    folder / 'assets'
    for folder in [Path.cwd(), Path.cwd() / 'Пересдача', Path.cwd().parent]
    if (folder / 'assets' / 'lab04_single_0.json').exists()
)

for index in range(2):
    response_file = assets_dir / f'lab04_single_{index}.json'
    received_json = json.loads(response_file.read_text(encoding='utf-8'))
    print(f'\nJSON ответа API: пример {index + 1}')
    print('Полный ответ:', response_file.resolve())
    print(json.dumps(received_json if SHOW_FULL_JSON else json_preview(received_json),
                     ensure_ascii=False, indent=2))
    for image in received_json['choices'][0]['message'].get('images', []):
        image_url = image['image_url']['url']
        if image_url.startswith('data:'):
            print('Поле url содержит data URL: изображение уже внутри ответа в Base64. GET не нужен.')
        else:
            print('Поле url содержит сетевую ссылку:', image_url)


## Вывод

Создано изображение по шаблону. Сравнены двухэтапная проверка с генерацией и условная генерация одним запросом.